In [6]:
#room specific code only activates when using parser
#Try to make keys easier to make.
#try to make [room].addDescription(n-, "description") mean that after n times, that becomes the default text, unless there is a description located past it.
#add ability to add synonyms to things.
#improve NPCs.
#NPC walking is broken, and I don't know why.
#make additional interactions for objects work with more than one word.
from just_playback import Playback
import climage, os, jaraco, appdirs, sys
class NPC:
    def __init__(self, name, description, startingLocation, **kwargs):
        self.name = name
        self.description = description
        self.location = startingLocation
        startingLocation.npcs.append(self)
        self.dialogue = {}
        self.placeInPath = 0
        self.properties = []
        self.pathingDirs = []
        self.listed = True
        #changes properties if listed in arguments, otherwise creates new property
        for item in kwargs:
            if("listed" in item):   
                self.listed = kwargs["listed"]
            if("path" in  item):
                path = kwargs["path"]
                
            else:
                self.properties.append(item)
    def addDialogue(self, num, key, text):
        self.dialogue[str(num)] = [key, text]
    def getDialogue(self, num):
        if num in self.dialogue.keys():
            return f"\nThey say: {self.dialogue[str(num)][1]}"
    def remDialogue(self, num):
        if num in self.dialogue.keys():
            del self.dialogue[str(num)]
    def moveNPC(self, newRoom, npc):
        """Moves an npc to another room. Takes in the old room, new room, then potentially who will show up."""
        self.location.npcs.remove(self)
        newRoom.npcs.append(npc)
        self.location = newRoom
    def setupPath(self, pathingDirs):
        """Makes the path, if there is an Xn at the end, that part is multiplied n times. If the direction is empty, then the npc stays."""
        newList = []
        if len(pathingDirs) == 1:
            #If the length is only 1 item
            if "X" in pathingDirs[0]:
                #Checks if you are multiplying the specified direction
                for item in range(int(pathingDirs[0][pathingDirs[0].find("X")+1:])):
                    newList.append(pathingDirs[0][0:pathingDirs[0].find("X")])
                self.pathingDirs = newList 
                return
            else:
                self.pathingDirs = pathingDirs
                return
        #If the length is greater than 1 item.
        for item in pathingDirs:
            if("X" in item):
               #Checks if you are multiplying the specified direction
               for item in range(int(item[item.find("X")+1:])):
                    newList.append(item[0:item.find("X")])
            else:
               newList.append(item)
        self.pathingDirs = newList
    def moveInPath(self):
        """Progresses the NPC forward in their path, if they have one"""
        position =self.placeInPath % len(self.pathingDirs)
        for room, direction in self.location.exits:
            if direction == self.pathingDirs[position]:
                self.moveNPC(room, self)
                self.placeInPath+=1
                return
    def talk(self):
        """A loop that allows for you to talk to this NPC without using the parser."""
        if not (len(self.dialogue.items()) > 0):
            return
        while True:
            for num, key in self.dialogue.items():
                print(f"\n{num}. {key[0]}")
            text = input("What do you want to ask? Type a number. Pres 'q' to quit.\n")
            if (text == 'q'):
                return 
            print(self.getDialogue(text))
            self.remDialogue(text)
        return
    @staticmethod
    def moveAll(npcs):
        for npc in npcs:
            npc.moveInPath()
class Player:
    def __init__(self):
        self.inventory = []
        self.combinations = []
        self.understandings = {}
    def addUnderstanding(self, text, word):
        self.understandings[text] = word
    def getUnderstanding(self, text):
        return self.understandings[text]
    def addToInventory(self, thing):
        self.inventory.append(thing)
    def removeFromInventory(self, thing):
        self.inventory.remove(thing)
    def getInventory(self):
        """Returns the player's inventory in the form of a numerated list."""
        things = ""
        if(len(self.inventory) == 0):
            return "\n You have nothing."
        if len(self.inventory)>0:
            things += "\nYou have: "
            for num, x in enumerator(self.inventory):
                things += "\n" + num + ". " + x.name
            return things
        else:
            return ""
    def saveGame(self, text):
        """Creates a file with all your inputs up to that point."""
        if(os.path.exists("save.txt")):
            os.remove("save.txt")
            with open("save.txt", "w") as file:
                for line in text:
                    file.write(line + "\n")
        else: 
            with open("save.txt", "w") as file:
                for line in text:
                    file.write(line + "\n")
    def useSave(self):
        """Plays the save file as if the user input the commands."""
        with open("save.txt", "r") as file:
            while True:
                line = file.readline()
                if not line:
                    return ""
                yield line
                
                
                
                
    def addCombination(self,thing, toCombine, output, oneWay=False):
        self.combinations.append([thing, toCombine, output])
        if(oneWay == False):
            self.combinations.append([toCombine, thing, output])
    def getCombination(self, thing, toCombine):
        """Returns the combination and removes the items involved."""
        for combination in self.combinations:
            if thing in combination and toCombine in combination:
                if thing in self.inventory and toCombine in self.inventory:
                    self.inventory.remove(thing)
                    self.inventory.remove(toCombine)
                    self.inventory.append(combination[2])
                    return f"\nYou've gotten: {combination[2].name}"
            return "That didn't work"
class Thing:
    def __init__(self, name, description, **kwargs):
        self.name = name
        self.description = description
        self.use = ""
        self.things = []
        self.properties = []
        self.interactions = []
        self.portable = True
        self.transparent = True
        self.listed = True
        #changes properties if listed in arguments, otherwise creates new property
        for item in kwargs:
            if("portable" in item):    
                self.portable = kwargs["portable"]
            elif("transparent" in item):   
                self.transparent = kwargs["transparent"]     
            elif("listed" in item):   
                self.listed = kwargs["listed"]
            else:
                self.properties.append(item)
    def addDefUse(self, use):
        self.use = use
    def getDefUse(self):
        return "\n"+self.use
    def addInteraction(self, word, text):
        """adds an additional parser action that doesn't do anything."""
        self.interactions.append([word, text])
    def getInteraction(self, word):
        """gets an additional parser action from the list of added actions."""
        for interaction in self.interactions:
            if interaction[0] == word:
                return interaction[1]
        return ""
    def addThing(self, thing):
        self.things.append(thing)
    def removeThing(self, thing):
        self.things.remove(thing)
    def getThings(self) -> str:
        """Lists everything listable within this thing."""
        listed = 0
        unlisted = 0
        for thing in self.things:
            if thing.listed == True:
                listed += 1
            else:
                unlisted +=1
        if listed == 0 and unlisted > 0:
            return ""
        """Returns a string containing a list of items contained within an object."""
        string = f"\nThere are these things around {self.name}: "
        if(len(self.things) > 0):   
            for x in self.things:
                string += x.name + " "
            return string
        else:
            return ""
    

class Room:
    def __init__(self, name, firstDescription):
        self.name = name
        self.descriptions = []
        self.descriptions.append([0, firstDescription])
        self.exits = []
        self.things = []
        self.witnessed = 0
        self.special = False
        self.hint = "\nThere are no extra hints for this room."
        self.image = ""
        self.npcs = []
        self.musicPath = ""
    def addMusic(self, fileName):
        fileName = resource_path(fileName)
        if(fileName.count("\\") > 1):
            self.musicPath = fileName
            return
        self.musicPath = fileName
    def playMusic(self):
        if os.path.isfile(self.musicPath):
            if playback.playing:
                playback.stop()
                playback.load_file(self.musicPath)
                playback.set_volume(0.5)
                playback.play()
            else:
                playback.load_file(self.musicPath)
                playback.set_volume(0.5)
                playback.play()
    @staticmethod
    def revDirection(direction):
        """Takes a direction and reverses it, for use in a Room method. If you want more directions, add them and their inverse here."""
        source = {"north":"south",#regular cardinal directions
                  "east":"west",
                  "up":"down",#spacial directions
                  "in":"out",
                  "north-west":"south-east",#extra cardinal directions 
                  "north-east":"south-west",}
        if direction in source.keys():
            return source[direction]
        if direction in source.values():
            for key, value in source.items():
                if direction == value:
                    return key
                
        
        
    def addAdjoiningRoom(self, room, direction, oneWay=False):
        """Adds a room by specifying where you come from, where you're going, and in what direction.
        Additionally, you can specify the oneWay argument to be true in order to get a 1-way door."""
        self.exits.append([room, direction])
        if oneWay == False:
            room.exits.append([self, Room.revDirection(direction)])
    def addThing(self, thing):
        """Adds the object to the list of objects in the room."""
        self.things.append(thing)
    def removeThing(self, thing):
        """Removes the object from the list of objects in the room."""
        self.things.remove(thing)
    
    def getThings(self):
        """Returns a list of names of the objects in the location. If there are none, it returns an empty string."""
        listed = 0
        unlisted = 0
        for thing in self.things:
            
            if thing.listed == True:
                listed += 1
            else:
                unlisted +=1
        for npc in self.npcs:
            if npc.listed == True:
                listed+=1
            else:
                unlisted+=1
        if listed == 0 and unlisted > 0:
            return ""
        #the above is a test to see whether there are only unlisted things in the room, which would then not be mentioned.
        names = ""
        if(len(self.things) > 0 or len(self.npcs) > 0):
            for thing in self.things:
                if(thing.listed == True):
                    names += " " + thing.name + ", "
            for npc in self.npcs:
                if(npc.listed == True):
                    names+= " " + npc.name + ", "
            return "\nYou see: " + names
        else:
            return ""
    def incrementWitnessed(self):
        self.witnessed += 1
    def getWitnessed(self):
        return self.witnessed
    def addDescription(self, witnessed, description):
        """Adds a description for a given value or values of times having visited a room, or if you've visited a room more than n times."""
        if (type(witnessed) == type("text")):
            self.special = True
        if(self.special):
            if("-" in witnessed):
                #when adding a range of descriptions
                nums = witnessed.split("-")
                a, b = map(lambda n: int(n), nums)
                for num in range(a,b+1):
                    self.descriptions.append([num, description])
                self.special = False
                return
        else:
            self.descriptions.append([witnessed, description])
    def getDescription(self, number):
        """Returns the specific description asked for in the number argument."""
        #how do I make it return the elongated description until the number has been matched?
                
        #finds your additional room description, if it exists.
        #number has been matched
        for num, desc in self.descriptions:
            if num == number:
                return desc
        #final case, returns the default room description
        #number doesn't have a match.
        return self.descriptions[0][1]
    def changeDescription(self, number, newDescription):
        for num, desc in self.descriptions:
            if num == number:
                self.descriptions[0][1] = newDescription
                return 
        raise Exception("You input an invalid description identifier.")
    def setHint(self, hint):
        self.hint = hint
    def getHint(self):
        return "\n"+self.hint
    def addImage(self, fileName):
        fileName = resource_path(fileName)
        if(fileName.count("\\") > 1):
            self.image = climage.convert(fileName, is_unicode=True)
            return
        self.image = climage.convert(fileName, is_unicode=True)          
    def getImage(self):
        if self.image != "":
            return self.image
        return ""
    
def parser(room, user_input, player, rooms=[], things=[], npcs=[]):
    text = user_input
    NPC.moveAll(npcs)
    if(type(user_input) == type(["apple"])):
        user_input = user_input[-1]
    """Takes user input and converts it into 2 keywords that then are matched against possible commands."""
    if(len(user_input.split()) == 1):
        #a single word case.
        user_input = user_input.lower().strip()
        container = {"n":"north","s":"south","e":"east","w":"west",
                     "out":"out","in":"in",
                     "nw":"north-west","ne":"north-east","sw":"south-west","se":"south-east"}
        if(user_input in container.keys()):
            #trying to move in a direction using a shorthand of the direction
            for exit in room.exits:
                if(container[user_input] == exit[1]):
                    exit[0].incrementWitnessed()
                    return exit[0], exit[0].getDescription(exit[0].getWitnessed()) + exit[0].getThings()
        if(user_input == "i" or user_input == "inventory"):
            return room, player.getInventory()
        
        if(user_input == "help"):
            return room, """You can MOVE/GO 'LOCATION', LOOK 'AROUND'/'OBJECT'/'ITEMS', GET 'ITEM' or USE 'ITEM' or USE 'item' ON 'another item',and can 'QUIT GAME' If you REALLY need a hint, then GET HINT should help. \n\nThe directions are mostly cardinal (those on a compass). In addition to the directions, you could use just the first letter, or first 2 letters for extra cardinal directions. You could also type i or inventory to view your inventory. You can save with SAVE"""
        if(user_input == "save"):
            player.saveGame(text[:-1])
            return room, "\nYou have saved the game. To play this save, exit then return and run USE SAVE."
    if len(user_input.split()) == 2:
        #a double word case.
        first, second = user_input.lower().strip().split()
        
        
        if(first not in ["use","look", "get", "help", "move", "talk", "quit"]):
            """Code which returns the additional interactions."""
            for item in player.inventory:
                if item.name == second:
                    return room, item.getInteraction(first)
            for item in room.things:
                if item.name == second:
                    return room, item.getInteraction(first)
        #room specific code and use code--------------------------------------------------------------------------
        #ideas: teleporation, keys + use, using items together, locked doors, being in a certain room changing all rooms.
        
        
        
        
        if(first == "use"):
            #specific object/room interactions-------------------------------------------
            #for example, checking if you're in a specific room before doing something.
            if(second=="key" and room.name == "glowing room" and key in player.inventory):
                player.removeFromInventory(key)
                glowing_room.addAdjoiningRoom(locked_room, "north")
                glowing_room.changeDescription(0, "There is very little glowing here, but there is a door opening north which you opened. There is a path south-west")
                start.changeDescription(0, "It seems your journey is complete.")
                return room, "You unlocked the door, it is north of you.\n" + room.getDescription(room.getWitnessed()) + room.getThings() 
            
            #end of specific object/room interactions------------------------------------
            for x in player.inventory:
                #display default use
                if x.name == second:
                    return room, x.getDefUse()
            for x in room.things:
                if x.name == second:
                    return room, x.getDefUse()
            if(second == "save"):
#add the ability to replay the game up to the point you stopped.
                player.useSave()
            return room, "You can't do that here with that."
        #room specific and use code------------------------------------------------------------------------------
        
        
        if(first == "look"):
            """Checks if your first choice was to look at something, then finds the correct text to output for what you typed."""
            if(second in (lambda n: n.name, room.npcs)):
                for npc in room.npcs:
                    if npc.name == second:
                        return room, npc.description
            if(second == "around" or second == "room"):
                return room, room.getDescription(room.getWitnessed()) + room.getThings()
            elif (second == "inventory" or second == "items"):
                return room, player.getInventory()
            else:
                #if the object is in the player's inventory
                if second in player.getInventory():
                      for x in player.inventory:
                        if x.name.lower() == second:
                            return room, x.description

                #if the object isn't in the player's inventory.
                else:
                    for x in room.things:
                        if x.name.lower() == second:
                            if x.portable:
                                #if you can pick up the object 
                                return room, x.description + x.getThings()
                            else:
                                #if you can't pick up the object
                                return room, x.description + x.getThings()
                    return room, f"{second} isn't here."

        if(first == "get"):
            """Manages the user acquiring items, and removing that same item from the room it was in to put it in the inventory."""
            if second == "hint":
                return room, room.getHint()
            for x in room.things:
            #gets an item if it isn't in another item and you can pick it up.
                if second == x.name:
                    #if you can pick up the item
                    if(x.portable):
                        player.addToInventory(x)
                        room.removeThing(x)
                        return room, "\nYou got it."
                    else:
                        return room, "\nYou can't pick that up."
            for thing in room.things:
            #gets an item if it is in another item and you can pick it up.   
                for x in thing.things:
                    if second == x.name:
                        #if you can pick up the item
                        if(x.portable):
                            player.addToInventory(x)
                            thing.removeThing(x)
                            return room, "\nYou got it."
                        else:
                            return room, "\nYou can't pick that up."
            return room, "That's either not here or you can't pick that up."
            

        if(first == "help" and second == "me"):
            """Checks if your choice was to get help, which then returns some advice on playing the game."""
            return room, """You can MOVE/GO 'LOCATION', LOOK 'AROUND'/'OBJECT'/'ITEMS', GET 'ITEM' or USE 'ITEM' or USE 'item' ON 'another item',and can 'QUIT GAME' If you REALLY need a hint, then GET HINT should help. \n\nThe directions are mostly cardinal (those on a compass). In addition to the directions, you could use just the first letter, or first 2 letters for extra cardinal directions. You could also type i or inventory to view your inventory. You can save with SAVE"""

        if(first == "move" or first == "go"):
            """Checks if your choice was to move, then it looks through all the exits of the room
            to see if there is a matching exit to your move."""
            for exit in room.exits:
                if(second == exit[1]):
                    exit[0].incrementWitnessed()
                    return exit[0], exit[0].getDescription(exit[0].getWitnessed()) + exit[0].getThings()
            else:
                return room, "\n You can't move in that direction here."
        if(first == "quit" and second == "game"):
            return room, "quit"
        return room, "You input an invalid command. Try 'look'ing or 'move'ing"       
    
    
    elif (len(user_input.split()) > 2):
#remember to add LOOK "at" OBJECT as a possibility.
        """If the length of the input is greater than 2"""
        for key, word in player.understandings.items():
            """Allows for additional understandings to be used, then converted into other forms"""
            if(key in user_input):
                return parser(room, player.getUnderstanding(user_input), player, rooms, things)
        words = user_input.lower().strip().split()
        if (words[0] == "use" and words[2] == "on"):
            "Allows for the combining of items."
            thing1 = words[1]
            thing2 = words[3]
            for thing in player.inventory:
                if thing1 == thing.name:
                    thing1 = thing
                if thing2 == thing.name:
                    thing2 = thing
            if type(thing1) == type("String") or type(thing2) == type("String"):
                return room, "Those things aren't in your inventory"
            else:
                return room, player.getCombination(thing1, thing2)
        if(words[0] == "look" and words[1] == "at"):
            for npc in room.npcs:
                if npc.name == words[2]:
                    return room, npc.description
            for thing in player.inventory:
                if thing.name == words[2]:
                    return room, thing.description + thing.getThings()
            for thing in room.things:
                if thing.name == words[2]:
                    return room, thing.description + thing.getThings()
            if(words[2] == room.name):
                return room, room.getDescription(room.getWitnessed()) + room.getThings()
        if(words[0] == "talk" and words[1] == "to"):
            for npc in room.npcs:
                if words[2] == npc.name:
                    npc.talk()
                    return room, room.getDescription(room.getWitnessed()) + room.getThings()
        return room, "\nI didn't understand that."
    else:
        return room, "\nI didn't understand that."
def resource_path(relative_path, forPyinstaller = True):
    """A function I stole to be able to run smaller executables"""
    if(not forPyinstaller):
        return relative_path
    try:
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)
player = Player()
playback = Playback()
#"code"-------------------------------------------------------------------------------------- 
#You can write room specific or use interactions code in the parser function
#
#ROOMS----
#The place you start in must always be called "start"
#add rooms with: [room] = Room("room", "description")
#join rooms with: [room].addAdjoiningRoom([destination], "direction_text") add a OneWay=True or just "True" to the end to make it a one way.
#add new description when seen n times: [room].addDescription(n, "description"), n can be a range "a-b"
#add music for the room with: [room].addMusic(fileName)
#add image for the room with: [room].addImage(fileName)
#OBJECTS----
#add object: [object] = Thing("object", "description", portableState(defaults to true), transparentState(defaults to true), listed(default is true))
#join object to room: [room].addThing([object])
#add use attempt text: [object].addDefUse("text for use")
#add interaction text: [object].addInteraction("action", "text to display")
#add object to object (like a table containing things) [object].addThing([otherObject])
#HINTS
#You can do [room].setHint("text") to add a hint that could be used.
#PLAYER
#you could add the ability to merge items through [player].addCombination([item1], [item2], [output]) it automatically adds the other direction, unless you add True or oneWay=True
#You can add an understanding of a certain bunch of text as a set of simpler commands, with [player].addUnderstanding(textToUnderstand, newText)
#NPCs
#you can create NPCs with [npc] = NPC("name", "description", [startingLocation]"
#you can give dialogue with: [name].addDialogue(num, question, answer)
#---------------------------------------------------------------------------------------------
openingText = "Welcome to the wonderful search! I'd advice you move around and check your list at least twice. \nif you need help, which you probably do, just type 'help me' or 'help'\n\n\n"
start = Room("Strange location", "You are in the starting room. There is an exit north")
start.setHint("There is an open box here, person sized. You could GO IN, if you wanted.")
box = Room("box", "You went into the box, and it seems like there are extra rooms north and north-west, somehow. Although, you went a bit deep, so you'd need a ladder to get back out.")
box.addDescription("2-3", "You can see the opening of the box from here.")
start.addAdjoiningRoom(box, "in", oneWay=True)
player.addUnderstanding("go in box", "go in")
player.addUnderstanding("go into box", "go in")
player.addUnderstanding("get in box", "go in")
player.addUnderstanding("get into box", "go in")
player.addUnderstanding("look in box", "look box")
box.addImage("files/box.png")
start.addMusic("files/background.ogg")
box.addMusic("files/box.ogg")

test = Room("Empty Room", "You can see John's marching tracks on the ground.")
start.addAdjoiningRoom(test, "north")


john = NPC("john", "He is a balding, middle-aged man, yet he still looks fit despite the wrinkles.", start)
john.addDialogue(1, "what's going on?", "You know, I don't know either how we're here.")
john.addDialogue(2, "who are you?", "I'm just a bike repairman; just bicycles, not moterbikes. *sighs*")
john.addDialogue(3, "why is this happening?", "I suspect it has to do with some weird space-time phenomenon.")
john.addDialogue(4, "How have you been sustaining yourself?", "Food? Oh, I've been surviving off of the carpet and whatever moisture leaks through that door. *points at door* I can't quite tell what it is, though.")
john.addDialogue(5, "What do you remember?", "I don't remember how I got in here. I could have been asleep, awake, driving, drinking, etc.")
john.addDialogue(6, "Why are you moving back and forth?", "I'm doing my cardio.")
john.setupPath(["north", "south"])

box_thing = Thing("box", "it looks like its big enough to get in", portable=False)
box_thing.addInteraction("pick", "You want to pick up the box? Why? It's a big box!")
box_thing.addInteraction("open", "The box is open")
box_thing.addInteraction("close", "This achieved nothing. The box is closed.")
box_thing.addInteraction("kick", "You kicked the box, nothing happened.")
box_thing.addInteraction("rip", "This box is sturdy, so you can't even bend it's sides.")
start.addThing(box_thing)
start.addImage("files/start.png")

hat_rack = Thing("hatrack", "It looks perfect for holding some hats", portable=False)
hat_rack.addDefUse("Your hat is already sitting perfectly still and at home in this hat rack.")
hat_rack.addInteraction("kick", "You kick the hatrack, but nothing happens as you set it back upright.")
start.addThing(hat_rack)

door = Thing("door", "it looks like a good door, perhaps its made of mahogany", portable=False)
door.addDefUse("You tried the handle, but it wouldn't budge.")
door.addInteraction("kick", "You kicked the door, but it wouldn't budge.")
door.addInteraction("open", "The door won't open.")
start.addThing(door)


box_north = Room("corridor", "the light is fading from your view, but up ahead you see pure darkness.")
glowing_room = Room("glowing room", "The light fades, but a bluer glow seems to come from the walls. There is a path south-east.")
glowing_room.setHint("This room might become brighter if you come back, try paying attention then.")
glowing_room.addDescription(2, "The light fades, but a strong blue glow seems to come from the walls.")
glowing_room.addDescription(3, "The light fades, but a staggering blue glow seems to come from the walls, making apparent a keyhole in the wall, before the light goes away. You remember the location of the keyhole.")

box.addAdjoiningRoom(box_north, "north")
box.addAdjoiningRoom(glowing_room, "north-west")

locked_room = Room("A locked room", "You have entered this locked room, so you get a point.")
box_to_start = Thing("box", "This box looks like it could fit you.", listed=True)
locked_room.addThing(box_to_start)
locked_room.addAdjoiningRoom(start, "in", oneWay=True)
#script: join to glowing_room if you have key

pure_darkness = Room("Pure darkness", "There is nothing, there is no light, but you feel some things around you. You feel a small hole east of you, at least you think?")
hole = Room("Little hole", "You are on your belly, trying to get through this hole. You feel like a Ted right now.")
hole.setHint("You could pick up stuff, or try again and see if you notice something the next time.")
hole.addDescription("2-50", "You've tried sthis before in this hole, so you know your way around better. You felt something small just at this point. Maybe it's a key?")
pure_darkness.addAdjoiningRoom(hole, "east")
box_north.addAdjoiningRoom(pure_darkness, "north")
pure_darkness.addImage("files/pure_darkness.png")

key = Thing("key", "This is a shiny golden key.", listed=False)
key.addDefUse("There's a time and a place, but it's not here.")
hole.addThing(key)

#end of "code" ------------------------------------------------------------------------------



#Runs the game
output = "" 
#edit this print if you want a unique start text.
print(openingText)
area = start
moves = 0
text = []
try:
    print(area.getImage())
    area.playMusic()
    print(area.name.upper() + f", MOVES: {moves}" + "\n" + area.getDescription(0), area.getThings())
    while output != "quit":
        text.append(input("-What do you want to do?"))
        if(text[0] == "use save"):
            for line in player.useSave():
                area, output = parser(area, line.strip().lower(), player, rooms=[glowing_room, locked_room],things=[key])
                moves+=1
                print(area.getImage())
                area.playMusic()
                print(area.name.upper() + f", MOVES: {moves}"+ "\n" + output.strip())
        print("\n\n")
        area, output = parser(area, text, player, rooms=[glowing_room, locked_room], things=[key], npcs=[john])
        moves+=1
        print(area.getImage())
        area.playMusic()
        print(area.name.upper() + f", MOVES: {moves}"+ "\n" + output.strip())
except KeyboardInterrupt:
    if playback.active:
        playback.stop()
        

Welcome to the wonderful search! I'd advice you move around and check your list at least twice. 
if you need help, which you probably do, just type 'help me' or 'help'



▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄
▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄